In [ ]:
!unzip "/content/drive/MyDrive/kaggle/FungiCLEF25/train_category.zip" -d "/content/"
!unzip "/content/drive/MyDrive/kaggle/FungiCLEF25/val_category.zip" -d "/content/"

!mkdir /content/train_val_combine
!mv /content/category /content/train_category

!cp /content/train_category/*/*.JPG /content/train_val_combine
!cp /content/val_category/*/*.JPG /content/train_val_combine

In [ ]:
# === Imports and Setup ===
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import backend as K
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization, Concatenate, GlobalAveragePooling2D
from tensorflow.keras.models import Model
import numpy as np
import pandas as pd
from PIL import ImageFile
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import os

from tensorflow.keras import mixed_precision
# Enable mixed precision globally
mixed_precision.set_global_policy("mixed_float16")

ImageFile.LOAD_TRUNCATED_IMAGES = True

# === Constants ===
IMAGE_SIZE = (300,300)
BATCH_SIZE = 2
NUM_CLASSES = 2427
IMAGE_DIR = "/content/train_val_combine"

# === Load CSVs ===
df = pd.read_csv('/content/FungiTastic-FewShot-Train.csv')
val_df = pd.read_csv('/content/FungiTastic-FewShot-Val.csv')
test_df = pd.read_csv('/content/FungiTastic-FewShot-Test.csv', index_col='observationID')

df = pd.concat([df, val_df])
df["category_id"] = df["category_id"].astype(str)
train_category_id = df["category_id"].values
df = df.drop('observationID', axis=1)

In [ ]:
# === Preprocessing Functions ===
def required_col(dataframe, test_df):
    return dataframe[test_df.columns]

def drop_col(dataframe):
    return dataframe.drop(['eventDate', 'year', 'month', 'day', 'district', 'coorUncert'], axis=1)

def label_encode(dataframe):
    label_cols = ['habitat', 'countryCode', 'hasCoordinate', 'substrate', 'region', 'metaSubstrate', 'biogeographicalRegion']
    dataframe[label_cols] = dataframe[label_cols].apply(LabelEncoder().fit_transform)
    return dataframe

def split_train_test_data(df):
    class_counts = df['category_id'].value_counts()
    rare_classes = class_counts[class_counts == 1].index
    rare_class_df = df[df['category_id'].isin(rare_classes)]
    rest_df = df[~df['category_id'].isin(rare_classes)]

    train_df_rest, val_df = train_test_split(
        rest_df, test_size=0.2, stratify=rest_df['category_id'], random_state=42
    )
    train_df = pd.concat([train_df_rest, rare_class_df], axis=0)
    train_df = train_df.sample(frac=1, random_state=42).reset_index(drop=True)
    return train_df, val_df

# replacing nan values with median value of that class
def nan_insertion(dataframe):
    num_cols = dataframe.select_dtypes(include=[np.number]).columns
    for col in num_cols:
        dataframe[col] = dataframe.groupby("category_id")[col].transform(lambda x: x.fillna(x.median()))
    return dataframe

In [ ]:
# === Apply Preprocessing ===
df = required_col(df, test_df)
df = drop_col(df)
df = label_encode(df)
df["category_id"] = train_category_id

df = nan_insertion(df)
df["elevation"] = df["elevation"].fillna(df["elevation"].mean())
df["landcover"] = df["landcover"].fillna(df["landcover"].mean())

train_df_full, val_df_full = split_train_test_data(df)

# === Scale Tabular Data ===
scaler = StandardScaler()
train_tabular = train_df_full.drop(['filename', 'category_id'], axis=1)
val_tabular = val_df_full.drop(['filename', 'category_id'], axis=1)
train_tabular_scaled = scaler.fit_transform(train_tabular)
val_tabular_scaled = scaler.transform(val_tabular)

In [ ]:
# === Image Generators ===
datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.4,
    horizontal_flip=True,
    vertical_flip=True,
    fill_mode='nearest'
)

train_generator = datagen.flow_from_dataframe(
    train_df_full, directory=IMAGE_DIR, x_col="filename", y_col="category_id",
    target_size=IMAGE_SIZE, batch_size=BATCH_SIZE, class_mode="sparse", has_ext=True, shuffle=True, seed=42
)

val_generator = datagen.flow_from_dataframe(
    val_df_full, directory=IMAGE_DIR, x_col="filename", y_col="category_id",
    target_size=IMAGE_SIZE, batch_size=BATCH_SIZE, class_mode="sparse", has_ext=True, shuffle=False
)

# Found 8201 validated image filenames belonging to 2427 classes.
# Found 1903 validated image filenames belonging to 1391 classes.

In [ ]:
# === Custom Generator for Image + Tabular ===
def custom_generator(image_gen, tabular_features, filename_to_index):
    while True:
        image_batch, label_batch = next(image_gen)
        label_batch = tf.convert_to_tensor(label_batch, dtype=tf.int32)  # ✅ key fix

        batch_filenames = image_gen.filenames[image_gen.batch_index * image_batch.shape[0] - image_batch.shape[0]:image_gen.batch_index * image_batch.shape[0]]

        batch_tabular = []
        for name in batch_filenames:
            if name in filename_to_index:
                idx = filename_to_index[name]
                batch_tabular.append(tabular_features[idx])
            else:
                batch_tabular.append(np.zeros(tabular_features.shape[1]))

        batch_tabular = np.array(batch_tabular)

        if batch_tabular.shape[0] != image_batch.shape[0]:
            continue

        yield (
            {"image_input": image_batch, "tabular_input": batch_tabular},
            label_batch
        )


# Create mapping from filename to index in tabular features
filename_to_index = {fname: i for i, fname in enumerate(train_df_full['filename'])}

# train_gen = custom_generator(train_generator, train_df_full, train_tabular_scaled, BATCH_SIZE)
train_gen = custom_generator(train_generator, train_tabular_scaled, filename_to_index)

val_filename_to_index = {fname: i for i, fname in enumerate(val_df_full['filename'])}
val_gen = custom_generator(val_generator,  val_tabular_scaled, val_filename_to_index)



In [ ]:
from tensorflow.keras.callbacks import Callback

# To save best model in every 5 epochs
class SaveBestEvery5Epochs(Callback):
    def __init__(self, save_path='best_model.keras',  save_dir='checkpoints'):
        super().__init__()
        self.best_val_acc = 0
        self.save_path = save_path
        self.save_dir = save_dir

    def on_epoch_end(self, epoch, logs=None):
        current_val_acc = logs.get('val_top_5_accuracy')
        if current_val_acc is None:
            return

        # Every 5 epochs
        if (epoch + 1) % 5 == 0:
            if current_val_acc > self.best_val_acc:
                print(f"\nEpoch {epoch+1}: val_accuracy improved from {self.best_val_acc:.5f} to {current_val_acc:.5f}. Saving model.")
                self.best_val_acc = current_val_acc
                # self.model.save(self.save_path)
                filename = f"model_epoch_{epoch+1}_valtop5_{current_val_acc:.4f}.keras"
                save_path = os.path.join(self.save_dir, filename)
                self.model.save(save_path)

            else:
                print(f"\nEpoch {epoch+1}: val_accuracy did not improve from {self.best_val_acc:.5f}. Model not saved.")


In [ ]:
# In this we gradually unfreeze layer instead of directly unfreezing all layer in once
class BlockWiseGradualUnfreeze(Callback):
    def __init__(self, base_model, lr_schedule, freeze_epochs=5, unfreeze_epochs=20, final_epochs=5, blocks_per_unfreeze=5):
        super().__init__()
        self.base_model = base_model
        self.lr_schedule = lr_schedule
        self.freeze_epochs = freeze_epochs
        self.unfreeze_epochs = unfreeze_epochs
        self.final_epochs = final_epochs
        self.blocks_per_unfreeze = blocks_per_unfreeze
        self.total_blocks = len(self.base_model.layers) // self.blocks_per_unfreeze
        self.unfrozen_blocks = 0

    def on_epoch_begin(self, epoch, logs=None):
        if epoch < self.freeze_epochs:
            print(f"🧊 Keeping base model frozen for warm-up (epoch {epoch + 1}/{self.freeze_epochs}).")
            self.base_model.trainable = False

        elif epoch < self.freeze_epochs + self.unfreeze_epochs:
            print(f"🔵 Epoch {epoch + 1} begins...")
            blocks_to_unfreeze = min(self.unfrozen_blocks + 1, self.total_blocks)

            print(f"✅ Unfreezing {self.blocks_per_unfreeze} layers...")
            start_index = -blocks_to_unfreeze * self.blocks_per_unfreeze
            for layer in self.base_model.layers[start_index:]:
                layer.trainable = True

            print(f"📜 {len(self.base_model.layers) - blocks_to_unfreeze * self.blocks_per_unfreeze} layers remaining to unfreeze.")

            self.unfrozen_blocks = blocks_to_unfreeze

            # Re-compile and rebuild the training function
            self.model.compile(
                optimizer=tf.keras.optimizers.AdamW(
                    learning_rate=self.lr_schedule,
                    weight_decay=1e-4,
                    epsilon=1e-7,
                ),
                loss=self.model.loss,
                metrics=self.model.metrics
            )
            self.model.make_train_function()

        else:
            print(f"🔥 Fully unfreezing base model for fine-tuning (epoch {epoch + 1}/{self.freeze_epochs + self.unfreeze_epochs + self.final_epochs}).")
            self.base_model.trainable = True
            self.model.compile(
                optimizer=tf.keras.optimizers.AdamW(
                    learning_rate=self.lr_schedule,
                    weight_decay=1e-4,
                    epsilon=1e-7,
                ),
                loss=self.model.loss,
                metrics=self.model.metrics
            )
            self.model.make_train_function()


In [ ]:
from tensorflow.keras import backend as K
from keras.metrics import top_k_categorical_accuracy
from tensorflow.keras.saving import register_keras_serializable
from tensorflow.keras.losses import Loss
from sklearn.utils.class_weight import compute_class_weight

import math
from tensorflow.keras.optimizers.schedules import CosineDecayRestarts

@register_keras_serializable()

# To mininze effect of imbalance classes
class FocalLoss(Loss):
    def __init__(self, gamma=2.0, alpha=0.25, **kwargs):
        super().__init__(**kwargs)
        self.gamma = gamma
        self.alpha = alpha

    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.int32)
        y_true = tf.one_hot(y_true, depth=NUM_CLASSES)
        y_pred = tf.clip_by_value(y_pred, tf.keras.backend.epsilon(), 1. - tf.keras.backend.epsilon())
        cross_entropy = -y_true * tf.math.log(y_pred)
        weight = self.alpha * tf.pow(1 - y_pred, self.gamma)
        loss = tf.reduce_sum(weight * cross_entropy, axis=-1)
        return loss

# === Custom Metric ===
@register_keras_serializable()
def top_5_accuracy(y_true, y_pred):
    y_true = tf.cast(y_true, tf.int32)
    y_true = tf.one_hot(y_true, depth=NUM_CLASSES)
    y_pred = tf.cast(y_pred, tf.float32)
    return top_k_categorical_accuracy(y_true, y_pred, k=5)

In [ ]:
# === Model Architecture ===
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.applications import DenseNet121

# Image Branch
image_input = Input(shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3), name="image_input")

# base_model = EfficientNetB3(weights='imagenet', include_top=False, input_tensor=image_input)
base_model = DenseNet121(include_top=False, weights='imagenet', input_tensor=image_input)
# base_model = DenseNet121(include_top=False, weights='imagenet', input_shape=(*IMAGE_SIZE, 3))

base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation="relu")(x)
x = Dropout(0.3)(x)
cnn_output = x

# Tabular Branch
tabular_input = Input(shape=(train_tabular_scaled.shape[1],), name="tabular_input")
y = Dense(128, activation="relu")(tabular_input)
y = Dropout(0.3)(y)
y = BatchNormalization()(y)
y = Dense(256, activation="relu")(y)
y = Dropout(0.3)(y)
y = BatchNormalization()(y)
tabular_output = y

# Merge + Output
merged = Concatenate()([cnn_output, tabular_output])
z = Dense(512, activation="relu")(merged)
z = Dropout(0.3)(z)
z = BatchNormalization()(z)
output = Dense(NUM_CLASSES, activation="softmax", dtype='float32')(z)

model = Model(inputs=[image_input, tabular_input], outputs=output)


In [ ]:
# === Compile ===
steps_per_epoch = len(train_generator)

lr_schedule = CosineDecayRestarts(
    initial_learning_rate=1e-3,    # Start high! (was ~1e-3 usually)
    first_decay_steps=steps_per_epoch * 5,  # Restart every 5 epochs
    t_mul=2.0,                     # Double the cycle length after each restart
    m_mul=0.8,                     # Reduce max LR slightly after each restart
    alpha=1e-5                     # Never let LR fall below this
)

# # class_weight_dict can used but it will over focus imbalance class, as we are already using Focal loss to handle it.
# y_train_labels = train_df_full["category_id"].astype(int).values
# class_weights = compute_class_weight(
#     class_weight='balanced',
#     classes=np.unique(y_train_labels),
#     y=y_train_labels
# )
# class_weight_dict = {label: weight for label, weight in zip(np.unique(y_train_labels), class_weights)}


ModelCheckpoint = [
    SaveBestEvery5Epochs(save_dir=r'/content/drive/MyDrive/kaggle/FungiCLEF25/')
]

callback = BlockWiseGradualUnfreeze(
    base_model=base_model,
    lr_schedule=lr_schedule,
    freeze_epochs=1,
    unfreeze_epochs=20,
    final_epochs=5,
    blocks_per_unfreeze=5
)

# === Train ===
model.fit(
    train_gen,
    steps_per_epoch=len(train_generator),
    validation_data=val_gen,
    validation_steps=len(val_generator),
    epochs=40,
    callbacks= [callback,
                ModelCheckpoint]
    # class_weight=class_weight_dict
)



We can also manually control instead of using callbacks which gives more control over training

In [ ]:
def unfreeze_layers(base_model, num_layers_to_unfreeze):
    """
    Unfreezes the last `num_layers_to_unfreeze` layers in the base model.
    Freezes all other layers.
    """
    total_layers = len(base_model.layers)

    for i, layer in enumerate(base_model.layers):
        if i < total_layers - num_layers_to_unfreeze:
            layer.trainable = False
        else:
            layer.trainable = True

In [ ]:
# === Phase 1: Freeze base model for first 5 epochs ===
print("🔁 Phase 1: Frozen training for 5 epochs")
unfreeze_layers(base_model, 0)  # Freeze all layers
model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=lr_schedule, weight_decay=1e-4, epsilon=1e-7),
    loss=FocalLoss(gamma=2.0, alpha=0.25),
    # loss=focal_loss(gamma=2.0, alpha=0.25),
    metrics=["accuracy", top_5_accuracy]
)


model.fit(
    train_gen,
    steps_per_epoch=steps_per_epoch,
    validation_data=val_gen,
    validation_steps=len(val_generator),
    epochs=5,
    callbacks=[SaveBestEvery5Epochs(save_dir=r'/content/drive/MyDrive/kaggle/FungiCLEF25/')],
)

# === Phase 2: Gradual unfreezing over 20 epochs ===
print("\n🔁 Phase 2: Gradual unfreezing for 20 epochs")
total_layers = len(base_model.layers)
fine_tune_epochs = 25
unfreeze_every_n_epochs = 2
layers_per_step = max(1, total_layers // (fine_tune_epochs // unfreeze_every_n_epochs))

for step in range(0, fine_tune_epochs):
    epoch_num = 8 + step  # start from epoch 5

    if step % unfreeze_every_n_epochs == 0:
        # Every n epochs, unfreeze more layers
        blocks_to_unfreeze = (step // unfreeze_every_n_epochs + 1)
        layers_to_unfreeze = min(blocks_to_unfreeze * layers_per_step, total_layers)
        unfreeze_layers(base_model, layers_to_unfreeze)
        print(f"\n🔓 Epoch {epoch_num+1}: Unfreezing last {layers_to_unfreeze} layers")

        model.compile(
              optimizer=keras.optimizers.AdamW(learning_rate=lr_schedule, weight_decay=1e-4, epsilon=1e-7),
              loss=FocalLoss(gamma=2.0, alpha=0.25),
              # loss=focal_loss(gamma=2.0, alpha=0.25),
              metrics=["accuracy", top_5_accuracy]
          )

    model.fit(
        train_gen,
        steps_per_epoch=steps_per_epoch,
        validation_data=val_gen,
        validation_steps=len(val_generator),
        epochs=epoch_num + 1,
        initial_epoch=epoch_num,
        callbacks=[SaveBestEvery5Epochs(save_dir=r'/content/drive/MyDrive/kaggle/FungiCLEF25/')],
    )

# === Phase 3: Final 5 epochs with all layers trainable ===
print("\n🚀 Phase 3: Final training with all layers trainable (5 epochs)")
unfreeze_layers(base_model, total_layers)
model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=lr_schedule, weight_decay=1e-4, epsilon=1e-7),
    loss=FocalLoss(gamma=2.0, alpha=0.25),
    # loss=focal_loss(gamma=2.0, alpha=0.25),
    metrics=["accuracy", top_5_accuracy]
)

model.fit(
    train_gen,
    steps_per_epoch=steps_per_epoch,
    validation_data=val_gen,
    validation_steps=len(val_generator),
    epochs=30,
    initial_epoch=25,
    callbacks=[SaveBestEvery5Epochs(save_dir=r'/content/drive/MyDrive/kaggle/FungiCLEF25/')],
)


# ✅ Evaluation + Save Predictions Code

In [ ]:
from tensorflow.keras.models import load_model
from tensorflow.keras.utils import custom_object_scope
from tensorflow.keras.losses import Loss
from tensorflow.keras.saving import register_keras_serializable
from tensorflow.keras.applications.efficientnet import preprocess_input as efficientnet_preprocess
import os
from tqdm import tqdm
import pandas as pd

from tensorflow.keras import backend as K

@register_keras_serializable()
class FocalLoss(Loss):
    def __init__(self, gamma=2.0, alpha=0.25, **kwargs):
        super().__init__(**kwargs)
        self.gamma = gamma
        self.alpha = alpha

    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.int32)
        y_true = tf.one_hot(y_true, depth=NUM_CLASSES)
        y_pred = tf.clip_by_value(y_pred, tf.keras.backend.epsilon(), 1. - tf.keras.backend.epsilon())
        cross_entropy = -y_true * tf.math.log(y_pred)
        weight = self.alpha * tf.pow(1 - y_pred, self.gamma)
        loss = tf.reduce_sum(weight * cross_entropy, axis=-1)
        return loss

# === Custom Metric ===
from keras.metrics import top_k_categorical_accuracy
@register_keras_serializable()
def top_5_accuracy(y_true, y_pred):
    y_true = tf.cast(y_true, tf.int32)
    y_true = tf.one_hot(y_true, depth=NUM_CLASSES)
    y_pred = tf.cast(y_pred, tf.float32)
    return top_k_categorical_accuracy(y_true, y_pred, k=5)

# Load best saved model
custom_objects = {
    'FocalLoss': FocalLoss,
    'top_5_accuracy': top_5_accuracy
}
# with custom_object_scope(custom_objects):
#     model = load_model('/content/drive/MyDrive/kaggle/FungiCLEF25/best_model.keras', custom_objects=custom_objects)
model1 = load_model('/content/drive/MyDrive/kaggle/FungiCLEF25/model_epoch_8.keras', custom_objects=custom_objects)

# === Preprocess Test Data ===
test_df = pd.read_csv('/content/FungiTastic-FewShot-Test.csv', index_col='observationID')
test_df = required_col(test_df, test_df)
test_df = drop_col(test_df)
test_df = label_encode(test_df)
test_df.head()
# test_df = nan_insertion(test_df)
test_df["elevation"] = test_df["elevation"].fillna(test_df["elevation"].mean())
test_df["landcover"] = test_df["landcover"].fillna(test_df["landcover"].mean())


test_tabular_scaled = scaler.transform(test_df.drop(['filename'], axis=1))
test_image_paths = test_df['filename'].values

# === Prediction Loop ===
top_k = 10
predictions = []
obs_ids = []
seen_obs_ids = set()
for i, filename in tqdm(enumerate(test_image_paths), total=len(test_image_paths)):
    obs_id = test_df.index[i]
    if obs_id in seen_obs_ids:
        continue  # skip duplicate obs_id
    seen_obs_ids.add(obs_id)

    # Load and preprocess image
    img_path = os.path.join(r"/content/fullsize", filename)
    img = tf.keras.preprocessing.image.load_img(img_path, target_size=IMAGE_SIZE)
    img_array = tf.keras.preprocessing.image.img_to_array(img)
    img_array = efficientnet_preprocess(img_array)
    img_array = np.expand_dims(img_array, axis=0)

    # Prepare tabular input
    tabular_array = np.expand_dims(test_tabular_scaled[i], axis=0)

    # Predict
    preds = model1.predict({"image_input": img_array, "tabular_input": tabular_array}, verbose=0)
    top_preds = np.argsort(preds[0])[::-1][:top_k]  # Get top 10

    print(filename,top_preds)

    predictions.append(" ".join([str(x) for x in top_preds]))
    obs_ids.append(test_df.index[i])

# === Save to CSV ===
submission_df = pd.DataFrame({
    "observationId": obs_ids,
    "predictions": predictions
})
# observationId,predictions result csv column

submission_df = submission_df.drop_duplicates(subset=["observationId"])

submission_df.to_csv("top10_predictions.csv", index=False)
print("✅ Saved top-10 predictions to top10_predictions.csv")
